## Data Download

Download images from a WMS server and export YOLO-formatted segmentation masks for a geo-dataset of polygons. Points can be used for the image download but will result in faulty segmentation masks.

### Dependencies

In [1]:
import os

import geopandas as gpd
from owslib.wms import WebMapService

from detect_wildlife_crossings.modelling.helpers import polygon_to_yolo_segmentation
from detect_wildlife_crossings.wms.helpers import get_crop_bboxes

### Config

In [12]:
SOURCE_GPKGS = ["../data/geodata/CH_wildlife_crossings_osm_bridges_only.gpkg"]
CRS = "EPSG:3035"  # ETRS89 / LAEA Europe, for import and WMS request
OUTPUT_DIR = "../data/satellite_images/"
PIXEL_RESOLUTION_M = 2  # 2 meters per pixel
BUFFER_SIZE = 640  # Target image size in pixels (will be BUFFER_SIZE x BUFFER_SIZE)

In [13]:
WMS_URL = "https://image.discomap.eea.europa.eu/arcgis/services/GioLand/VHR_2021_LAEA/ImageServer/WMSServer/?request=GetCapabilities&service=WMS"

### WMS Connection

In [14]:
wms = WebMapService(
    WMS_URL,
    version="1.3.0",
)
layer_name = list(wms.contents)[0]
print(layer_name)

VHR_2021_LAEA


#### Execution

In [15]:
for f in SOURCE_GPKGS:
    print(f"Processing source geometry file: {os.path.basename(f)}")
    gdf = gpd.read_file(f)
    gdf = gdf[
        (gdf.geom_type == "Polygon")
        | (gdf.geom_type == "Point") & (gdf["bridge"] == "yes")
    ]
    print(f"Number of geometries to process: {len(gdf)}")

    gdf_centroids = gdf.to_crs(epsg=CRS.split(":")[1]).centroid
    areas_of_interest = get_crop_bboxes(gdf_centroids, BUFFER_SIZE)

    output_subdir = os.path.join(
        OUTPUT_DIR,
        os.path.splitext(os.path.basename(f))[0] + "_" + str(BUFFER_SIZE),
    )
    os.makedirs(output_subdir, exist_ok=True)
    os.makedirs(os.path.join(output_subdir, "images"), exist_ok=True)
    os.makedirs(os.path.join(output_subdir, "labels"), exist_ok=True)

    # for every point create a square around it and request a WMS crop centered on the point
    img_format = "image/jpeg"
    img_size = (
        int(BUFFER_SIZE * 2 / PIXEL_RESOLUTION_M),
        int(BUFFER_SIZE * 2 / PIXEL_RESOLUTION_M),
    )  # width, height in pixels

    for idx, bbox in enumerate(areas_of_interest):
        # Request WMS image
        img = wms.getmap(
            layers=[layer_name],
            srs=CRS,
            bbox=bbox,
            size=img_size,
            format=img_format,
            transparent=True,
        )

        # Save image to file
        img_data = img.read()
        img_filename = os.path.join(output_subdir, "images", f"crossing_{idx}.png")
        with open(img_filename, "wb") as f:
            f.write(img_data)

        # save polygon label to file in yolo format
        label_filename = os.path.join(output_subdir, "labels", f"crossing_{idx}.txt")
        polygon = gdf.geometry.iloc[idx]

        # Convert polygon to YOLO segmentation format
        yolo_line = polygon_to_yolo_segmentation(polygon, bbox, class_id=0)

        if yolo_line:
            with open(label_filename, "w") as f:
                f.write(yolo_line + "\n")
            print(f"Saved WMS crop and label for crossing {idx} to {img_filename}")
        else:
            print(f"Warning: Could not convert geometry {idx} to YOLO format")

Processing source geometry file: CH_wildlife_crossings_osm_bridges_only.gpkg
Number of geometries to process: 12
Saved WMS crop and label for crossing 0 to ../data/satellite_images/CH_wildlife_crossings_osm_bridges_only_640\images\crossing_0.png
Saved WMS crop and label for crossing 1 to ../data/satellite_images/CH_wildlife_crossings_osm_bridges_only_640\images\crossing_1.png
Saved WMS crop and label for crossing 2 to ../data/satellite_images/CH_wildlife_crossings_osm_bridges_only_640\images\crossing_2.png
Saved WMS crop and label for crossing 3 to ../data/satellite_images/CH_wildlife_crossings_osm_bridges_only_640\images\crossing_3.png
Saved WMS crop and label for crossing 4 to ../data/satellite_images/CH_wildlife_crossings_osm_bridges_only_640\images\crossing_4.png
Saved WMS crop and label for crossing 5 to ../data/satellite_images/CH_wildlife_crossings_osm_bridges_only_640\images\crossing_5.png
Saved WMS crop and label for crossing 6 to ../data/satellite_images/CH_wildlife_crossings